In [ ]:
from dotenv import load_dotenv
from anthropic import Anthropic
import json
%pip install -U anthropic

load_dotenv()

client = Anthropic()
model ="claude-sonnet-4-5"

In [ ]:
def add_user_message(messages, text):
    user_message = ({"role": "user", "content": text})
    messages.append(user_message)

def add_assistant_message(messages, text):
    assitant_message = ({"role": "assistant", "content": text})
    messages.append(assitant_message)

def chat(messages, system = None, temperature = 1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "extra_body": {"temperature": temperature},
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    response = client.messages.create(**params)
    return "".join(block.text for block in response.content if block.type == "text")

# Building the Core Functions
The evaluation pipeline consists of three main functions, each with a specific responsibility. Let's start with the simplest one - the function that handles individual prompts.

## The run_prompt Function
This function takes a test case and merges it with our prompt template.
Right now, we're keeping the prompt extremely simple. We're not including any formatting instructions, so Claude will likely return more verbose output than we need. We'll refine this later as we iterate on our prompt design.


In [ ]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result."""
    prompt = f"""
Please solve the following task:
{test_case["task"]}
"""
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

## The run_test_case
This function orchestrates running a single test case and grading the result.
For now, we're using a hardcoded score of 10. The grading logic is where we'll spend significant time in upcoming sections, but this placeholder lets us test the overall pipeline.

In [ ]:
def run_test_case(test_case):
    """ Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # TODO - Grading
    score = 10

    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }


## The run_eval Function
This function processes every test case in our dataset and collects all the results into a single list.

In [ ]:
def run_eval(dataset):
    """Loads the dataset and call run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    return results


## Running the Evaluation
To execute our evaluation pipeline, we load our dataset and run it through our functions
The first time you run this, expect it to take some time - even with Claude Haiku, it can take around 30 seconds to process a full dataset. We'll cover optimization techniques later.

In [ ]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

## Examining the Results

In [ ]:
print(json.dumps(results, indent=2))